# Exercise: Extracting Information from Emails with DSPy

In this exercise, you will build an intelligent email processing system with [DSPy](https://dspy.ai/). The system classifies emails, extracts structured entities, summarizes content, and recommends actions.

By the end, your pipeline will:

- **Classify email types** (order confirmation, support request, meeting invitation, etc.)
- **Extract key entities** (dates, amounts, product names, contact info)
- **Determine urgency levels** and required actions
- **Structure extracted data** into consistent formats
- **Handle multiple email formats** robustly

Run the scaffold cells first, then complete each `___` blank in the exercise cells below.

## Pipeline overview

```mermaid
flowchart TD
    rawEmail[rawEmail subject body sender] --> classify[ClassifyEmail]
    classify --> extract[ExtractEntities]
    extract --> summarize[SummarizeEmail]
    summarize --> actions[GenerateActionItems]
    classify --> actions
    extract --> actions
    actions --> result[dspy.Prediction]
    summarize --> result
```

## Setup

### Install dependencies

Before starting, install the required packages:

```bash
!pip install -qU dspy pydantic pyyaml
```


### MLflow DSPy Integration

Set up MLflow Tracing to understand what's happening under the hood.

<a href="https://mlflow.org/">MLflow</a> is an LLMOps tool that natively integrates with DSPy and offers explainability and experiment tracking. You can use MLflow to visualize prompts and optimization progress as traces to understand DSPy's behavior better.

![MLflow Trace](../assets/mlflow-tracing-email-extraction.png)

1. Install MLflow

```bash
%pip install mlflow>=3.0.0
```

2. Start MLflow UI in a separate terminal

```bash
mlflow ui --port 5000 --backend-store-uri sqlite:///mlruns.db
```

3. Connect the notebook to MLflow

```python
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("DSPy")
```

4. Enable tracing.

```python
mlflow.dspy.autolog()
```

To learn more, visit [MLflow DSPy Documentation](https://mlflow.org/docs/latest/llms/dspy/index.html).


## Step 1: Define Our Data Structures

First, define the types of information you want to extract from emails.

### Schema overview

```mermaid
classDiagram
    class EmailType {
        <<enumeration>>
        ORDER_CONFIRMATION
        SUPPORT_REQUEST
        MEETING_INVITATION
        NEWSLETTER
        PROMOTIONAL
        INVOICE
        SHIPPING_NOTIFICATION
        OTHER
    }
    class UrgencyLevel {
        <<enumeration>>
        LOW
        MEDIUM
        HIGH
        CRITICAL
    }
    class ExtractedEntity {
        +str entity_type
        +str value
        +float confidence
    }
```


In [ ]:
import dspy  # https://dspy.ai/
from enum import Enum
from typing import Optional
from pydantic import BaseModel  # https://docs.pydantic.dev/latest/


### Exercise 1a: `EmailType`

Use the schema diagram above. Fill in each enum value string.

- Include all eight email categories shown in the diagram.


In [ ]:
class EmailType(str, Enum):
    ORDER_CONFIRMATION = "___"
    SUPPORT_REQUEST = "___"
    MEETING_INVITATION = "___"
    NEWSLETTER = "___"
    PROMOTIONAL = "___"
    INVOICE = "___"
    SHIPPING_NOTIFICATION = "___"
    OTHER = "___"


<details><summary>Expected solution</summary>

```python
class EmailType(str, Enum):
    ORDER_CONFIRMATION = "order_confirmation"
    SUPPORT_REQUEST = "support_request"
    MEETING_INVITATION = "meeting_invitation"
    NEWSLETTER = "newsletter"
    PROMOTIONAL = "promotional"
    INVOICE = "invoice"
    SHIPPING_NOTIFICATION = "shipping_notification"
    OTHER = "other"
```
</details>


### Exercise 1b: `UrgencyLevel`

Define four urgency levels from low to critical.


In [ ]:
class UrgencyLevel(str, Enum):
    LOW = "___"
    MEDIUM = "___"
    HIGH = "___"
    CRITICAL = "___"


<details><summary>Expected solution</summary>

```python
class UrgencyLevel(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"
```
</details>


### Exercise 1c: `ExtractedEntity`

Create a Pydantic model with three fields: entity type, value, and confidence score.


In [ ]:
class ExtractedEntity(BaseModel):
    ___: ___
    ___: ___
    ___: ___


<details><summary>Expected solution</summary>

```python
class ExtractedEntity(BaseModel):
    entity_type: str
    value: str
    confidence: float
```
</details>


## Step 2: Create DSPy Signatures

Define one signature per pipeline stage. Each signature specifies inputs and outputs for a single LM call.


### Exercise 2a: `ClassifyEmail`

```mermaid
flowchart LR
    subgraph InputBox["Input"]
        direction TB
        A[email_subject<br>str]
        B[email_body<br>str]
        C[sender<br>str]
    end
    InputBox -- " " --> FUNC[ClassifyEmail<br>function]
    FUNC -- " " --> OutputBox["Output"]
    subgraph OutputBox["Output"]
        direction TB
        D[email_type<br>EmailType]
        E[urgency<br>UrgencyLevel]
        F[reasoning<br>str]
    end
```

Fill in `dspy.InputField` / `dspy.OutputField` and the field descriptions.


In [ ]:
class ClassifyEmail(dspy.Signature):
    """Classify the type and urgency of an email based on its content."""

    email_subject: str = dspy.___(desc="___")
    email_body: str = dspy.___(desc="___")
    sender: str = dspy.___(desc="___")

    email_type: EmailType = dspy.___(desc="___")
    urgency: UrgencyLevel = dspy.___(desc="___")
    reasoning: str = dspy.___(desc="___")


<details><summary>Expected solution</summary>

```python
class ClassifyEmail(dspy.Signature):
    """Classify the type and urgency of an email based on its content."""

    email_subject: str = dspy.InputField(desc="The subject line of the email")
    email_body: str = dspy.InputField(desc="The main content of the email")
    sender: str = dspy.InputField(desc="Email sender information")

    email_type: EmailType = dspy.OutputField(desc="The classified type of email")
    urgency: UrgencyLevel = dspy.OutputField(desc="The urgency level of the email")
    reasoning: str = dspy.OutputField(desc="Brief explanation of the classification")
```
</details>


### Exercise 2b: `ExtractEntities`

```mermaid
flowchart LR
    subgraph in [Inputs]
        c[email_content str]
        t[email_type EmailType]
    end
    subgraph out [Outputs]
        e[key_entities list ExtractedEntity]
        m[financial_amount Optional float]
        d[important_dates list str]
        i[contact_info list str]
    end
    in --> ExtractEntities --> out
```


In [ ]:
class ExtractEntities(dspy.Signature):
    """Extract key entities and information from email content."""

    email_content: str = dspy.___(desc="___")
    email_type: EmailType = dspy.___(desc="___")

    key_entities: list[ExtractedEntity] = dspy.___(desc="___")
    financial_amount: Optional[float] = dspy.___(desc="___")
    important_dates: list[str] = dspy.___(desc="___")
    contact_info: list[str] = dspy.___(desc="___")


<details><summary>Expected solution</summary>

```python
class ExtractEntities(dspy.Signature):
    """Extract key entities and information from email content."""

    email_content: str = dspy.InputField(desc="The full email content including subject and body")
    email_type: EmailType = dspy.InputField(desc="The classified type of email")

    key_entities: list[ExtractedEntity] = dspy.OutputField(
        desc="List of extracted entities with type, value, and confidence"
    )
    financial_amount: Optional[float] = dspy.OutputField(
        desc="Any monetary amounts found (e.g., '$99.99')"
    )
    important_dates: list[str] = dspy.OutputField(desc="List of important dates found in the email")
    contact_info: list[str] = dspy.OutputField(desc="Relevant contact information extracted")
```
</details>


### Exercise 2c: `GenerateActionItems`

```mermaid
flowchart LR
    subgraph in [Inputs]
        t[email_type EmailType]
        u[urgency UrgencyLevel]
        s[email_summary str]
        e[extracted_entities list ExtractedEntity]
    end
    subgraph out [Outputs]
        a[action_required bool]
        i[action_items list str]
        d[deadline Optional str]
        p[priority_score int]
    end
    in --> GenerateActionItems --> out
```


In [ ]:
class GenerateActionItems(dspy.Signature):
    """Determine what actions are needed based on the email content and extracted information."""

    email_type: EmailType = dspy.___()
    urgency: UrgencyLevel = dspy.___()
    email_summary: str = dspy.___(desc="___")
    extracted_entities: list[ExtractedEntity] = dspy.___(desc="___")

    action_required: bool = dspy.___(desc="___")
    action_items: list[str] = dspy.___(desc="___")
    deadline: Optional[str] = dspy.___(desc="___")
    priority_score: int = dspy.___(desc="___")


<details><summary>Expected solution</summary>

```python
class GenerateActionItems(dspy.Signature):
    """Determine what actions are needed based on the email content and extracted information."""

    email_type: EmailType = dspy.InputField()
    urgency: UrgencyLevel = dspy.InputField()
    email_summary: str = dspy.InputField(desc="Brief summary of the email content")
    extracted_entities: list[ExtractedEntity] = dspy.InputField(desc="Key entities found in the email")

    action_required: bool = dspy.OutputField(desc="Whether any action is required")
    action_items: list[str] = dspy.OutputField(desc="List of specific actions needed")
    deadline: Optional[str] = dspy.OutputField(desc="Deadline for action if applicable")
    priority_score: int = dspy.OutputField(desc="Priority score from 1-10")
```
</details>


### Exercise 2d: `SummarizeEmail`

```mermaid
flowchart LR
    subgraph in [Inputs]
        s[email_subject str]
        b[email_body str]
        k[key_entities list ExtractedEntity]
    end
    subgraph out [Outputs]
        m[summary str]
    end
    in --> SummarizeEmail --> out
```


In [ ]:
class SummarizeEmail(dspy.Signature):
    """Create a concise summary of the email content."""

    email_subject: str = dspy.___()
    email_body: str = dspy.___()
    key_entities: list[ExtractedEntity] = dspy.___()

    summary: str = dspy.___(desc="___")


<details><summary>Expected solution</summary>

```python
class SummarizeEmail(dspy.Signature):
    """Create a concise summary of the email content."""

    email_subject: str = dspy.InputField()
    email_body: str = dspy.InputField()
    key_entities: list[ExtractedEntity] = dspy.InputField()

    summary: str = dspy.OutputField(desc="A 2-3 sentence summary of the email's main points")
```
</details>


## Step 3: Build the Email Processing Module

### Exercise 3: `EmailProcessor`

Compose your four signatures into a single `dspy.Module`.

- Choose **`dspy.ChainOfThought`** or **`dspy.Predict`** for each submodule and use the same choice consistently.
- **ChainOfThought** adds an explicit reasoning field but costs an extra LM call per step.
- **Predict** is faster but skips the reasoning trace.

Wire the pipeline stages in `forward()` and return a single `dspy.Prediction` with all output fields.


In [ ]:
class EmailProcessor(dspy.Module):
    """A comprehensive email processing system using DSPy."""

    def __init__(self):
        super().__init__()

        self.classifier = dspy.___(ClassifyEmail)
        self.entity_extractor = dspy.___(ExtractEntities)
        self.action_generator = dspy.___(GenerateActionItems)
        self.summarizer = dspy.___(SummarizeEmail)

    def forward(self, email_subject: str, email_body: str, sender: str = ""):
        """Process an email and extract structured information."""

        # Step 1: Classify the email
        classification = self.classifier(
            email_subject=___,
            email_body=___,
            sender=___,
        )

        # Step 2: Extract entities from the full email content
        full_content = f"Subject: {email_subject}\n\nFrom: {sender}\n\n{email_body}"
        entities = self.entity_extractor(
            email_content=___,
            email_type=___,
        )

        # Step 3: Generate a concise summary
        summary = self.summarizer(
            email_subject=___,
            email_body=___,
            key_entities=___,
        )

        # Step 4: Determine required actions
        actions = self.action_generator(
            email_type=___,
            urgency=___,
            email_summary=___,
            extracted_entities=___,
        )

        # Step 5: Structure the results into a single Prediction
        return dspy.Prediction(
            email_type=___,
            urgency=___,
            summary=___,
            key_entities=___,
            financial_amount=___,
            important_dates=___,
            action_required=___,
            action_items=___,
            deadline=___,
            priority_score=___,
            reasoning=___,
            contact_info=___,
        )


<details><summary>Expected solution</summary>

```python
class EmailProcessor(dspy.Module):
    """A comprehensive email processing system using DSPy."""

    def __init__(self):
        super().__init__()

        # ChainOfThought adds a reasoning field; Predict is faster but skips explicit reasoning.
        self.classifier = dspy.ChainOfThought(ClassifyEmail)
        self.entity_extractor = dspy.ChainOfThought(ExtractEntities)
        self.action_generator = dspy.ChainOfThought(GenerateActionItems)
        self.summarizer = dspy.ChainOfThought(SummarizeEmail)

    def forward(self, email_subject: str, email_body: str, sender: str = ""):
        """Process an email and extract structured information."""

        # Step 1: Classify the email
        classification = self.classifier(
            email_subject=email_subject,
            email_body=email_body,
            sender=sender,
        )

        # Step 2: Extract entities from the full email content
        full_content = f"Subject: {email_subject}\n\nFrom: {sender}\n\n{email_body}"
        entities = self.entity_extractor(
            email_content=full_content,
            email_type=classification.email_type,
        )

        # Step 3: Generate a concise summary
        summary = self.summarizer(
            email_subject=email_subject,
            email_body=email_body,
            key_entities=entities.key_entities,
        )

        # Step 4: Determine required actions
        actions = self.action_generator(
            email_type=classification.email_type,
            urgency=classification.urgency,
            email_summary=summary.summary,
            extracted_entities=entities.key_entities,
        )

        # Step 5: Structure the results into a single Prediction
        return dspy.Prediction(
            email_type=classification.email_type,
            urgency=classification.urgency,
            summary=summary.summary,
            key_entities=entities.key_entities,
            financial_amount=entities.financial_amount,
            important_dates=entities.important_dates,
            action_required=actions.action_required,
            action_items=actions.action_items,
            deadline=actions.deadline,
            priority_score=actions.priority_score,
            reasoning=classification.reasoning,
            contact_info=entities.contact_info,
        )
```
</details>


## Step 4: Run the Email Processing System

### Exercise 4a: Configure the language model

Set your API key and configure DSPy with a language model. We suggest `openai/gpt-4o-mini`, but you can choose another supported model.

See the [DSPy language model guide](https://dspy.ai/learn/programming/language_models/).


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "___"
dspy.configure(lm=dspy.LM("___"))


### Load sample emails

Sample emails live in a YAML file under `../datasets/sample_emails.yaml`.


In [ ]:
from pathlib import Path
import yaml  # https://pyyaml.org/wiki/PyYAMLDocumentation

emails_path = Path("../datasets/sample_emails.yaml")
with emails_path.open(encoding="utf-8") as f:
    sample_emails = yaml.safe_load(f)["emails"]


### Process and display results

Run the pipeline on each sample email and print the key fields.


In [ ]:
processor = EmailProcessor()

print("🚀 Email Processing Demo")
print("=" * 50)

for i, email in enumerate(sample_emails):
    print(f"\n📧 EMAIL {i + 1}: {email['subject'][:50]}...")

    result = processor(
        email_subject=email["subject"],
        email_body=email["body"],
        sender=email["sender"],
    )

    print(f"   📊 Type: {result.email_type}")
    print(f"   🚨 Urgency: {result.urgency}")
    print(f"   📝 Summary: {result.summary}")

    if result.financial_amount:
        print(f"   💰 Amount: ${result.financial_amount:,.2f}")

    if result.action_required:
        print("   ✅ Action Required: Yes")
        if result.deadline:
            print(f"   ⏰ Deadline: {result.deadline}")
    else:
        print("   ✅ Action Required: No")


## Expected Output

Your output will vary slightly depending on the model, but it should resemble:

```
🚀 Email Processing Demo
==================================================

📧 EMAIL 1: Order Confirmation #12345 - Your MacBook Pro is on...
   📊 Type: order_confirmation
   🚨 Urgency: low
   📝 Summary: The email confirms John Smith's order #12345 for a MacBook Pro 14-inch in Space Gray, totaling $2,399.00, with an estimated delivery date of December 15, 2024. It includes a tracking number and contact information for customer support.
   💰 Amount: $2,399.00
   ✅ Action Required: No

📧 EMAIL 2: URGENT: Server Outage - Immediate Action Required...
   📊 Type: other
   🚨 Urgency: critical
   📝 Summary: The Site Reliability Team has reported a critical server outage that began at 2:30 PM EST, preventing all users from accessing the platform. They have requested the DevOps Team to join an emergency call immediately to address the issue.
   ✅ Action Required: Yes
   ⏰ Deadline: Immediately

📧 EMAIL 3: Meeting Invitation: Q4 Planning Session...
   📊 Type: meeting_invitation
   🚨 Urgency: medium
   📝 Summary: Sarah Johnson has invited the team to a Q4 planning session on December 20, 2024, from 2:00 PM to 4:00 PM EST in Conference Room A. Attendees are asked to confirm their participation by December 18th.
   ✅ Action Required: Yes
   ⏰ Deadline: December 18th
```


## Exercise 6: Add More Email Types

Extend the dataset to exercise `NEWSLETTER` and `PROMOTIONAL` classifications.

- Append one newsletter email and one promotional email to `../datasets/sample_emails.yaml` (or create `sample_emails_extended.yaml` and load that file instead).
- Re-run the processing loop above.
- Verify the classifier assigns `newsletter` and `promotional` types.


In [ ]:
# After adding entries to the YAML file, reload and re-run:
# with emails_path.open(encoding="utf-8") as f:
#     sample_emails = yaml.safe_load(f)["emails"]
# ... then run the processing loop again


<details><summary>Example newsletter and promotional YAML entries</summary>

```yaml
  - subject: "Weekly Tech Digest — December Edition"
    sender: "newsletter@techdigest.com"
    body: |
      Hello subscriber,

      Welcome to this week's Tech Digest! Here are the top stories:
      - AI agents are reshaping customer support workflows
      - New open-source tools for LLM evaluation

      Read the full articles on our website. You received this because you subscribed.

      Unsubscribe | Manage preferences

  - subject: "Flash Sale: 40% Off All Accessories — Today Only"
    sender: "deals@techstore.com"
    body: |
      Hi there,

      For one day only, save 40% on all laptop accessories — cases, docks, and cables.

      Use code FLASH40 at checkout. Offer expires tonight at midnight.

      Shop now and don't miss out!

      TechStore Marketing Team
```
</details>


## Next Steps

- **Add integration** with email providers (Gmail API, Outlook, IMAP)
- **Experiment with different LLMs** and optimization strategies
- **Add multilingual support** for international email processing
- **Optimization** for increasing the performance of your program

## Conclusion

You built a composed email processing pipeline with DSPy. The key ideas are:

- Define **signatures** that declare each LM task's inputs and outputs.
- Wrap signatures in **modules** (`ChainOfThought`, `Predict`) for reusable steps.
- Compose modules in a **`dspy.Module`** pipeline that returns a structured `Prediction`.

## References

- [Extracting Information from Emails with DSPy](https://dspy.ai/tutorials/email_extraction/)
